# Select Transfer Function


In [1]:
import hickle as hkl

from lib.utils import (
    G_sp_to_ctrl,
    input_names,
    output_labels,
    output_names,
    output_units,
)

tf = (2, 2)  # G_{3,3}: T3_FR

G_matrix = hkl.load("../outputs/G_simplified.hkl")
G_name = f"{output_names[tf[0]]}_{input_names[tf[1]]}"
G_ylabel = f"{output_labels[tf[0]]} / {output_units[tf[1]]}"
print(G_name)

G = G_sp_to_ctrl(G_matrix[tf])


T3_FR


# Simulate with noise


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

from lib.inputs import FR_0, Ff1_func, Ff2_func, Q1_func, Q2_func, Q3_func, T0_func
from lib.np_model import model
from lib.parameters import y0

base_funcs = [Ff1_func, Ff2_func, FR_0, Q1_func, Q2_func, Q3_func, T0_func]

# Aplica degrau apenas em FR
u = list(base_funcs)
u[2] = lambda t: FR_0 + 1.0

t = np.linspace(0, 2.5, 1500)

sol = solve_ivp(model, [t[0], t[-1]], y0, t_eval=t, args=(u,), rtol=1e-10, atol=1e-10)

# Tempo
t_og = sol.t

# Saída T3 (T1=0, T2=1, T3=2)
y_og = sol.y[2] - y0[2]

# Adiciona ruído
rng = np.random.default_rng(seed=42)
sigma = 0.02
noise = rng.normal(loc=0.0, scale=sigma, size=y_og.shape)
y_noisy = y_og + noise


# Get signal spectrum


In [3]:
from lib.plots import plot_fft

plot_fft(t_og, y_noisy, f"filters/{G_name}_fft")


Plot saved to ../figures/filters/T3_FR_fft.png


# Develop filters


In [4]:
import control as ctrl

# Frequencia de corte
wc = 40  # rad/h

# Filtro de primeira ordem
tal = 1 / wc
H1 = ctrl.TransferFunction([1], [tal, 1])
_, y_1filt = ctrl.forced_response(H1, T=t_og, U=y_noisy)

# Filtro de segunda ordem
wn = wc
zeta = 1 / np.sqrt(2)
H2 = ctrl.TransferFunction([wn**2], [1, 2 * zeta * wn, wn**2])
_, y_2filt = ctrl.forced_response(H2, T=t_og, U=y_noisy)


In [5]:
from lib.plots import plot_or_show, plt

omega = np.logspace(0, 3, 1000)

mag1, phase1, omega1 = ctrl.frequency_response(H1, omega)
mag2, phase2, omega2 = ctrl.frequency_response(H2, omega)

phase1 = np.unwrap(phase1)
phase2 = np.unwrap(phase2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Magnitude
ax1.semilogx(omega1, 20 * np.log10(mag1), color="red", label="Filtro de 1ª ordem")
ax1.semilogx(omega2, 20 * np.log10(mag2), color="black", label="Filtro de 2ª ordem")

ax1.grid(True, which="both")
ax1.set_ylabel("Magnitude / dB")
ax1.legend()

# Fase
ax2.semilogx(omega1, np.degrees(phase1), color="red", label="Filtro de 1ª ordem")
ax2.semilogx(omega2, np.degrees(phase2), color="black", label="Filtro de2ª ordem")

ax2.grid(True, which="both")
ax2.set_ylabel("Fase / (graus)")
ax2.set_xlabel("Frequência / (rad$\\cdot$h$^{-1}$)")
ax2.legend()

# FC
ax1.axvline(wc, color="blue", linestyle="--")
ax2.axvline(wc, color="blue", linestyle="--")
ax2.text(wc, ax2.get_ylim()[0], r"$\omega_c$", color="blue", ha="center", va="top")


plot_or_show(f"filters/{G_name}_bode")


Plot saved to ../figures/filters/T3_FR_bode.png


# Plot


In [6]:
fig = plt.figure(figsize=(8, 4))

plt.plot(t_og, y_noisy, color="blue")

plt.xlabel("Tempo / h")
plt.ylabel(G_ylabel)

plot_or_show(f"filters/{G_name}_noise")


Plot saved to ../figures/filters/T3_FR_noise.png


In [7]:
fig = plt.figure(figsize=(8, 4))

plt.plot(t_og, y_noisy, label="Sinal com ruído", color="blue", alpha=0.3)
plt.plot(
    t_og, y_1filt, label="Filtro passa-baixa de 1ª ordem", color="black", linewidth=2
)
plt.plot(
    t_og, y_2filt, label="Filtro passa-baixa de 2ª ordem", color="red", linewidth=2
)

plt.xlabel("Tempo / h")
plt.ylabel(G_ylabel)

plt.legend()
plot_or_show(f"filters/{G_name}_compare")


Plot saved to ../figures/filters/T3_FR_compare.png
